[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C62_Coding_Interview_Course/05_mock_drills/05_mock_drills.ipynb)

# 05 · 模拟面试与压力下的编码（时间预算 / 说出来 / 翻车点 / 采样三题 / 9 道 timed drill）

目标：本模块不教新算法，教**交付方式**。所有内容都做成可执行、可断言的形式——
包括那些看起来「软」的东西（时间预算、句式库、翻车点、评分表）。

本 notebook 你会亲手实现：
1. **45 分钟时间预算表**与**三档超时补救**（剩 10 / 5 / 2 分钟）的决策函数
2. **「说出来」句式库** + **30 秒沉默检测器**（对着自己的录屏时间线跑）
3. **七个翻车点的自动审计器**：含**复杂度口误检测**（对照真值表）、变量名黑名单、规格漂移比对
4. **蓄水池采样** R 算法 + 卡方检验；以及一个 off-by-one 版本——它让**第一个元素永远不可能被选中**
5. **加权采样**：A-Res 的 $u^{1/w}$ 与「前缀和 + 二分」两条路，各自卡方检验；
   以及一个「看起来按权重接受」实际几乎反过来的错误版本
6. **Fisher–Yates 与错误洗牌的偏差检验**：先用 $n!\nmid n^n$ 证明错误版本**必然**有偏，再用卡方把它抓出来
7. **9 道 timed drill**（题面 + 建议用时 + 频率/档位 + 评分要点 + 隐藏答案 + 对拍）
8. **五维评分表代码化**：为什么「代码全对但不说不测」会低于「代码 75% 但全程解说」

> 心智模型：**面试官不是在读你的代码，是在读「你的代码 + 你的解说」构成的那个系统。
> 每个评分维度上都必须留下证据——沟通的证据是解说，测试的证据是你跑过的用例。**

> 所有随机实验都用**固定种子**，所以下面每一个 assert 都是确定性的，不会偶发失败。

## 1 · 45 分钟的时间预算：把六步协议落到分钟

关键数字：**前 13 分钟一行代码都不写**（占 29%），写码窗口 17 分钟，自测 8 分钟。
下面把预算写成数据结构，并断言它**首尾相接、总长 45 分钟**——留空隙就说明你漏了一个阶段。

In [ ]:
import math, re, time, heapq
from collections import Counter, deque
import numpy as np

SEED = 20260817                      # 全局固定种子：保证所有统计检验的 assert 确定性
np.set_printoptions(precision=4, suppress=True)

# (阶段, 起始分钟, 结束分钟, 该阶段必须留下的"证据")
BUDGET = [
    ("澄清与复述",  0,  3, "输入规模 / 是否可改输入 / 异常输入怎么办"),
    ("举例走一遍",  3,  6, "5-8 元素的小例子 + 手算期望输出 + 边界清单"),
    ("暴力解口述",  6,  9, "一个一定正确的基线 + 它的复杂度（只说不写）"),
    ("最优思路",    9, 13, "优化依据（哪个重复计算被消掉）+ 新复杂度 + 取得同意"),
    ("写代码",     13, 30, "签名与返回 -> 主循环 -> 边界"),
    ("自测",       30, 38, "手跑正例并报中间状态 + 空/单元素"),
    ("收尾",       38, 45, "时间与空间复杂度 + 一个优化方向 + 反问一个问题"),
]

assert BUDGET[0][1] == 0 and BUDGET[-1][2] == 45
for (n1, s1, e1, _), (n2, s2, e2, _) in zip(BUDGET, BUDGET[1:]):
    assert e1 == s2, f"阶段之间有空隙或重叠: {n1} -> {n2}"   # 预算必须无缝覆盖 45 分钟

no_code_until = [e for n, s, e, _ in BUDGET if n == "最优思路"][0]
code_span     = [e - s for n, s, e, _ in BUDGET if n == "写代码"][0]
test_span     = [e - s for n, s, e, _ in BUDGET if n == "自测"][0]
print(f"不写代码的前置阶段共 {no_code_until} 分钟（占 {no_code_until/45:.0%}）")
print(f"写码窗口 {code_span} 分钟 · 自测窗口 {test_span} 分钟")
assert (no_code_until, code_span, test_span) == (13, 17, 8)

def budget_status(elapsed_min, stage_name):
    """返回 (状态, 超时分钟, 建议)。状态 ∈ {on_track, behind}。"""
    name, start, end, evidence = next(b for b in BUDGET if b[0] == stage_name)
    if elapsed_min <= end:
        return "on_track", 0, f"{name}: 还剩 {end - elapsed_min} 分钟，需交付「{evidence}」"
    over = elapsed_min - end
    return "behind", over, f"{name} 已超时 {over} 分钟 —— 立刻收口，进入下一阶段"

for e, st in [(2, "澄清与复述"), (16, "最优思路"), (33, "写代码")]:
    print(f"  t={e:>2}min  {budget_status(e, st)}")
assert budget_status(2, "澄清与复述")[0] == "on_track"
assert budget_status(16, "最优思路") == ("behind", 3, "最优思路 已超时 3 分钟 —— 立刻收口，进入下一阶段")
print("\n✅ 时间预算表自洽：7 个阶段无缝覆盖 45 分钟，且能对任意时刻给出落后判定。")

## 1b · 三档超时补救：剩 10 / 5 / 2 分钟

补救的原则是**换取部分分，而不是赌全分**。注意每一档都有一句「要说出口的原话」——
补救动作本身不值分，**说出来才值分**。

In [ ]:
RESCUE = {
    10: dict(
        action="砍掉所有优化，退回到一定写得完的版本（暴力解 / O(n log n) 而非 O(n)）",
        say="时间不多了，我先把 O(n^2) 的正确版本写完，写完再说怎么优化到 O(n)。",
        forbid="继续调你那个精妙的 O(n) 双指针边界"),
    5: dict(
        action="停手，开始测：用第②步的例子手跑一遍并报中间状态",
        say="我现在用刚才那个例子走一遍：i=0 时 window 是 ...",
        forbid="再加功能 / 再改变量名 / 再重构"),
    2: dict(
        action="把已知 bug 说清楚（在哪、为什么、怎么改），然后报复杂度",
        say="这里当 k 大于数组长度时会越界，入口加一句 k = min(k, len(a)) 即可，其余逻辑不受影响。",
        forbid="沉默地乱改；或者假装没问题"),
}

def rescue_plan(remaining_min):
    """剩余时间 -> 补救方案；时间充裕时返回 None（正常推进）。"""
    for t in sorted(RESCUE):                  # 2 -> 5 -> 10，取第一个覆盖当前剩余时间的档
        if remaining_min <= t:
            return t, RESCUE[t]
    return None

for r in [20, 8, 4, 1]:
    p = rescue_plan(r)
    tag = "正常推进" if p is None else f"启动【剩 {p[0]} 分钟】档"
    print(f"剩 {r:>2} 分钟 -> {tag}")
    if p:
        print(f"        做: {p[1]['action']}")
        print(f"        说: 「{p[1]['say']}」")

assert rescue_plan(20) is None
assert rescue_plan(8)[0] == 10 and rescue_plan(4)[0] == 5 and rescue_plan(1)[0] == 2
for t, d in RESCUE.items():
    assert d["say"] and d["action"] and d["forbid"]      # 每一档都必须有"说什么"
print("\n✅ 三档补救齐全。记住分界：写不完 -> 降级；写完没测 -> 停手去测；有 bug -> 说清楚。")

## 2 · 「说出来」句式库 + 30 秒沉默检测

think aloud 失败的原因几乎总是同一个：**试图直播思维**（紧张时的思维是碎片化的，说出来就是噪声）。
正确做法是**播报状态**——在固定时刻说固定的话。下面把它做成一张可查的表，
再写一个沉默检测器，对着自己录屏的「说话时刻」时间线跑。

In [ ]:
SCRIPTS = {
    "复述":       "我复述一下：给我 __，要我返回 __，其中 __ 保证 __。对吗？",
    "澄清规模":   "输入规模大概是 10^3 还是 10^7？这决定我用 O(n^2) 还是必须 O(n log n)。",
    "澄清边界":   "空输入返回什么？会有重复元素吗？可以修改输入数组吗？",
    "澄清返回":   "你要的是下标还是值？多解时返回任意一个还是全部？",
    "暴力解":     "最直接的做法是双层循环，O(n^2)。我先把它作为正确性基线。",
    "提优化":     "我注意到内层循环重复算了同一个前缀和，换成哈希表可以降到 O(n)。",
    "开始写":     "我先写签名和返回，然后主循环，最后补边界。",
    "报不变量":   "这个 left 指针的不变量是：[left, right) 内没有重复字符。",
    "报缺口":     "这一段处理主逻辑，空输入还没处理，我下面补。",
    "自测":       "我用 [3,1,4,1,5] 走一遍：i=0 时 ...，i=1 时 ...",
    "收尾":       "时间 O(n)、空间 O(k)，k 是不同字符数。要支持流式输入我会改成 ...",
    "卡住-降维":  "我先解一个更简单的版本：如果数组有序 / 如果 k=1，那么 __。放开这个约束后多出来的困难是 __。",
    "卡住-退回":  "我这条路要求 __ 成立，但反例 __ 说明它不成立，所以我退回 O(n^2) 先写出来。",
    "卡住-换视角": "我换个角度：这题等价于「按结束时间排序后的贪心」，我用后者写。",
    "接提示":     "哦，你是说 __（复述提示）。对，我刚才漏了 __ 这个性质，那么 __ 就可以用 __ 处理。",
    "申请安静":   "我需要 30 秒安静想一下这个边界，可以吗？",
}
print(f"句式库共 {len(SCRIPTS)} 条，覆盖阶段/卡住/接提示三类场景：\n")
for k in ["复述", "澄清规模", "报不变量", "卡住-退回", "接提示"]:
    print(f"  [{k}] {SCRIPTS[k]}")

assert len(SCRIPTS) >= 14
for group in ["卡住-降维", "卡住-退回", "卡住-换视角"]:      # 脱困话术必须三种都在
    assert group in SCRIPTS
assert all(s.strip().endswith(("？", "。", "...", "__")) for s in SCRIPTS.values())

def silence_gaps(speech_times_sec, total_sec, limit=30):
    """给定"你说话的时刻"（秒），返回所有超过 limit 秒的沉默区间。"""
    marks = [0.0] + sorted(float(t) for t in speech_times_sec) + [float(total_sec)]
    return [(a, b) for a, b in zip(marks, marks[1:]) if b - a > limit]

TOTAL = 45 * 60
bad_run  = [5, 20, 100, 140, 400, 1200, 1210, 2000]        # 典型"沉默型"面试
good_run = list(range(5, TOTAL, 25))                       # 每 25 秒有一次状态播报

gb = silence_gaps(bad_run, TOTAL)
print(f"\n沉默型: {len(gb)} 段超 30 秒的沉默，最长 {max(b-a for a,b in gb):.0f} 秒"
      f"（{max(b-a for a,b in gb)/60:.1f} 分钟）")
print(f"播报型: {len(silence_gaps(good_run, TOTAL))} 段")
assert len(gb) >= 4 and max(b - a for a, b in gb) > 600
assert silence_gaps(good_run, TOTAL) == []
print("✅ 沉默检测器可用。规则：超过 30 秒就必须出声——哪怕只是「我需要 30 秒安静想一下」。")

## 3 · 七个翻车点的自动审计器

七条里有五条（一上来就写 / 不问边界 / 写完不测 / 默默改需求 / 沉默超 30 秒）**与算法能力完全无关**。
下面把它们做成可对录屏跑的审计器。三个子检测值得单独看：

- **复杂度口误**：对照一张真值表，`sorted` 说成 O(n)、`in` 对 list 说成 O(1) 都会被抓
- **变量名黑名单**：从代码里抽出被赋值的名字，命中黑名单或无意义单字母就计数
- **规格漂移**：把「题目要求」和「你实现的」做成两个 dict 直接比对

In [ ]:
# ── 子检测 a：复杂度真值表（面试里说错这些是重创，因为它暗示你没有量级直觉）──
COMPLEXITY_TRUTH = {
    "sorted(a)":              "O(n log n)",
    "dict 查找（均摊）":       "O(1)",
    "dict 查找（最坏）":       "O(n)",
    "x in list":              "O(n)",
    "x in set":               "O(1)",
    "list.insert(0, x)":      "O(n)",
    "heapq.nlargest(k, a)":   "O(n log k)",
    "bisect 二分":            "O(log n)",
    "深度 n 的递归的空间":     "O(n)",
    "str 拼接 n 次（+=）":     "O(n^2)",
}

def check_complexity_claims(claims):
    """claims: {操作: 你说的复杂度} -> 返回说错的项 [(操作, 你说的, 正确的)]"""
    wrong = []
    for op, said in claims.items():
        truth = COMPLEXITY_TRUTH[op]
        if said.replace(" ", "") != truth.replace(" ", ""):
            wrong.append((op, said, truth))
    return wrong

# ── 子检测 b：变量名 ──
NAME_BLACKLIST = {"tmp", "tmp2", "temp", "res2", "aa", "bb", "kk", "xx", "foo", "data2", "l"}
OK_SHORT = {"i", "j", "k", "n", "m", "dp", "lo", "hi"}      # 约定俗成的短名可接受

def bad_variable_names(src):
    names = re.findall(r"^\s*([A-Za-z_]\w*)\s*=[^=]", src, flags=re.M)
    bad = [x for x in dict.fromkeys(names)
           if x in NAME_BLACKLIST or (len(x) <= 2 and x not in OK_SHORT)]
    return bad

# ── 子检测 c：规格漂移 ──
def spec_drift(spec, impl):
    return [(k, spec[k], impl.get(k, "<未实现>")) for k in spec if spec[k] != impl.get(k)]

demo_claims_wrong = {"sorted(a)": "O(n)", "x in list": "O(1)", "bisect 二分": "O(log n)"}
demo_src_bad = "tmp = 0\nl = []\nres2 = {}\nleft = 0\ncount_by_class = {}\n"
demo_spec = {"returns": "indices", "in_place": True, "stable": True}
demo_impl = {"returns": "values", "in_place": False, "stable": True}

print("复杂度口误:", check_complexity_claims(demo_claims_wrong))
print("坏变量名  :", bad_variable_names(demo_src_bad))
print("规格漂移  :", spec_drift(demo_spec, demo_impl))
assert len(check_complexity_claims(demo_claims_wrong)) == 2      # bisect 那条是对的
assert set(bad_variable_names(demo_src_bad)) == {"tmp", "l", "res2"}
assert [d[0] for d in spec_drift(demo_spec, demo_impl)] == ["returns", "in_place"]
assert check_complexity_claims({op: t for op, t in COMPLEXITY_TRUTH.items()}) == []
print("✅ 三个子检测就位。注意 str 的 += 拼接是 O(n^2)——这是解析类题目里最常见的复杂度口误。")

In [ ]:
PITFALLS = ["①一上来就写", "②不问边界", "③写完不测", "④复杂度说错",
            "⑤变量名混乱", "⑥默默改需求", "⑦沉默超30秒"]

def audit(session):
    """对一场面试做七项审计。session 是一个 dict（可从录屏时间线人工整理出来）。
    返回 {翻车点: 证据}，空 dict 表示七项全过。"""
    hits = {}
    gap = session["first_keystroke_sec"] - session["problem_read_done_sec"]
    if gap < 30:                                          # ① 题面念完 30 秒内就敲键盘
        hits[PITFALLS[0]] = f"念完题 {gap} 秒后就开始敲键盘"
    asked = set(session.get("clarifications", []))
    if not ({"边界", "空输入", "重复元素"} & asked):        # ② 边界一个字没问
        hits[PITFALLS[1]] = f"澄清里没有任何边界项（只问了 {sorted(asked)}）"
    if session.get("tests_run", 0) == 0:                  # ③ 一个用例都没跑
        hits[PITFALLS[2]] = "整场没有跑过任何用例"
    wrong = check_complexity_claims(session.get("complexity_claims", {}))
    if wrong:
        hits[PITFALLS[3]] = "; ".join(f"{o}: 说了{s}，实际{t}" for o, s, t in wrong)
    bad = bad_variable_names(session.get("code", ""))
    if len(bad) >= 3:                                     # ⑤ 三个以上无意义名字
        hits[PITFALLS[4]] = f"可疑变量名 {bad}"
    drift = spec_drift(session.get("spec", {}), session.get("impl", {}))
    if drift:
        hits[PITFALLS[5]] = "; ".join(f"{k}: 要求{a}，实际{b}" for k, a, b in drift)
    gaps = silence_gaps(session.get("speech_times", []), session.get("total_sec", TOTAL))
    if gaps:
        hits[PITFALLS[6]] = f"{len(gaps)} 段超 30 秒沉默，最长 {max(b-a for a,b in gaps):.0f} 秒"
    return hits

session_bad = dict(
    problem_read_done_sec=60, first_keystroke_sec=72,
    clarifications=["输出格式"], tests_run=0,
    complexity_claims=demo_claims_wrong, code=demo_src_bad,
    spec=demo_spec, impl=demo_impl, speech_times=bad_run, total_sec=TOTAL)

session_good = dict(
    problem_read_done_sec=60, first_keystroke_sec=60 + 13 * 60,   # 前 13 分钟不写代码
    clarifications=["规模", "边界", "空输入", "重复元素", "返回下标还是值"], tests_run=4,
    complexity_claims={"sorted(a)": "O(n log n)", "x in set": "O(1)"},
    code="left = 0\nright = 0\ncount_by_class = {}\nbest_iou = 0.0\n",
    spec=demo_spec, impl=dict(demo_spec), speech_times=good_run, total_sec=TOTAL)

hb, hg = audit(session_bad), audit(session_good)
print(f"翻车场次命中 {len(hb)}/7:")
for k, v in hb.items():
    print(f"  {k}: {v}")
print(f"\n达标场次命中 {len(hg)}/7 ✅")
assert len(hb) == 7 and len(hg) == 0
print("\n结论：七项里有五项只需要流程纪律。一次认真的录屏复盘，通常比多刷 50 道题更值。")

## 4 · 蓄水池采样：实现、归纳证明的数值验证，以及一个致命的 off-by-one

问题：流长 $n$ 未知，只能过一遍，内存只放 $k$ 个，要求每个元素最终留下的概率都是 $k/n$。

R 算法：前 $k$ 个直接入池；第 $i$ 个（0-based，$i\ge k$）以概率 $k/(i+1)$ 替换池中随机一个。
证明的关键是**裂项**：第 $t$ 个元素踢掉某个特定位置的概率是 $\frac{k}{t+1}\cdot\frac1k=\frac1{t+1}$，
于是「没被踢掉」是 $\frac{t}{t+1}$，连乘后逐项抵消，只剩 $\frac{k}{i+1}$。

下面用**卡方检验**验证。为了不依赖 scipy，临界值直接查表。

In [ ]:
# 卡方分布上尾临界值（只列本 notebook 用到的自由度），避免依赖 scipy
CHI2_CRIT = {(4, 0.05): 9.488,  (4, 0.001): 18.467,
             (5, 0.05): 11.070, (5, 0.001): 20.515,
             (9, 0.05): 16.919, (9, 0.001): 27.877,
             (10, 0.05): 18.307, (10, 0.001): 29.588,
             (23, 0.05): 35.172, (23, 0.001): 49.728}

def chi2_stat(obs, exp):
    obs, exp = np.asarray(obs, float), np.asarray(exp, float)
    assert np.all(exp > 5), "期望频数必须 > 5，否则卡方近似不成立（这是面试里可以说出来的细节）"
    return float(((obs - exp) ** 2 / exp).sum())

def chi2_test(obs, exp, alpha=0.001):
    """返回 (是否拒绝均匀/给定分布, 统计量, 临界值)。拒绝 = 分布有偏。"""
    df = len(obs) - 1
    stat, crit = chi2_stat(obs, exp), CHI2_CRIT[(df, alpha)]
    return stat > crit, stat, crit

def reservoir_one(stream, rng):
    """k=1 的蓄水池采样：第 i 个元素以 1/(i+1) 的概率成为当前选中者。"""
    chosen = None
    for i, x in enumerate(stream):
        if rng.integers(0, i + 1) == 0:          # 概率 1/(i+1)
            chosen = x
    return chosen

def reservoir_k(stream, k, rng):
    """通用蓄水池采样（Vitter R 算法）。返回 k 个元素，每个元素入选概率 k/n。"""
    pool = []
    for i, x in enumerate(stream):
        if i < k:
            pool.append(x)                       # 前 k 个直接入池
        else:
            j = int(rng.integers(0, i + 1))      # j 均匀落在 [0, i]
            if j < k:                            # 以概率 k/(i+1) 替换池中第 j 个
                pool[j] = x
    return pool

N, T = 11, 33000                                  # 11 个元素 -> 自由度 10（临界值表里有）
rng = np.random.default_rng(SEED)
cnt = np.zeros(N, dtype=int)
for _ in range(T):
    cnt[reservoir_one(range(N), rng)] += 1
rej, stat, crit = chi2_test(cnt, np.full(N, T / N))
print(f"k=1: 各元素被选次数 {cnt}  (期望 {T/N:.0f})")
print(f"     卡方统计量 {stat:.2f} < 临界值 {crit}（df=10, α=0.001） -> 不拒绝均匀 ✅")
assert not rej, (stat, crit)

K = 3
rng = np.random.default_rng(SEED + 1)
inc = np.zeros(N, dtype=int)
for _ in range(20000):
    for x in reservoir_k(range(N), K, rng):
        inc[x] += 1
freq = inc / 20000
print(f"\nk=3: 入选频率 {freq}  (理论 k/n = {K/N:.4f})")
print(f"     最大偏差 {np.abs(freq - K/N).max():.5f}")
assert np.abs(freq - K / N).max() < 0.02          # 6 sigma 以内，固定种子下确定通过
print("✅ 蓄水池采样的 k/n 性质在 k=1 与 k=3 上都成立。")

In [ ]:
def reservoir_one_offbyone(stream, rng):
    """错误版本：把 1/(i+1) 写成 1/i（0-based 与 1-based 混淆）。"""
    chosen = None
    for i, x in enumerate(stream):
        if i == 0 or rng.integers(0, i) == 0:     # 错：分母少了 1
            chosen = x
    return chosen

rng = np.random.default_rng(SEED + 2)
cnt_bad = np.zeros(N, dtype=int)
for _ in range(T):
    cnt_bad[reservoir_one_offbyone(range(N), rng)] += 1
print(f"off-by-one 版本的选中次数: {cnt_bad}")
print(f"  元素 0 被选中 {cnt_bad[0]} 次 —— 它的真实概率是 **0**")
rej, stat, crit = chi2_test(cnt_bad, np.full(N, T / N))
print(f"  卡方统计量 {stat:.1f} > 临界值 {crit} -> 拒绝均匀（有偏）✅")

# 手推：P(元素 j 存活) = (1/j) * prod_{t=j+1}^{n-1} (t-1)/t = 1/(n-1)，而 j=0 项含 (1-1/1)=0
assert cnt_bad[0] == 0, "元素 0 必然从不被选中：连乘里出现 (1 - 1/1) = 0"
assert rej and stat > 1000
tail = cnt_bad[1:] / T
print(f"  其余元素各约 {tail.mean():.4f}，理论 1/(n-1) = {1/(N-1):.4f}（而正确答案应是 {1/N:.4f}）")
assert abs(tail.mean() - 1 / (N - 1)) < 0.002
print("\n面试价值：这个 bug 不会报错、不会崩、小样本上看不出来 —— 只有统计检验能抓住它。")

## 5 · 加权采样：$u^{1/w}$ 与「前缀和 + 二分」两条路

两条路都要知道，因为它们的适用场景不同：

| 方法 | 复杂度 | 场景 |
|---|---|---|
| 前缀和 + 二分（`np.searchsorted`） | 预处理 O(n)，每次抽 O(log n) | **有放回**、权重固定、要抽很多次（如 dataloader 的类平衡采样） |
| **A-Res**：key $=u_i^{1/w_i}$，取最大的 $k$ 个 | 一遍 O(n log k) | **流式**、n 未知、**无放回** |

$k=1$ 时 A-Res 严格满足 $P(i)=w_i/\sum_j w_j$。$k>1$ 时**入选概率不再正比于 $w_i$**——
这是练习 2 要量化的陷阱，也是长尾重采样里真实的报告错误来源（见 C58-03）。

In [ ]:
def weighted_pick_ares(items, weights, rng):
    """A-Res (Efraimidis-Spirakis), k=1：给每个元素配 key = u^(1/w)，取 key 最大者。"""
    best, best_key = None, -1.0
    for x, w in zip(items, weights):
        if w <= 0:
            continue                                  # 权重 0 的元素永不入选
        key = rng.random() ** (1.0 / w)               # w 越大，key 越靠近 1
        if key > best_key:
            best_key, best = key, x
    return best

def weighted_sample_replace(weights, size, rng):
    """有放回加权采样：前缀和 + 二分。返回下标数组。"""
    cum = np.cumsum(np.asarray(weights, float))
    u = rng.random(size) * cum[-1]
    return np.searchsorted(cum, u, side="right")      # side='right' 保证边界归到右侧区间

W = [1.0, 2.0, 3.0, 4.0, 5.0]                          # 5 类 -> 自由度 4
p_true = np.array(W) / sum(W)

T2 = 50000
rng = np.random.default_rng(SEED + 3)
cnt = np.zeros(len(W), dtype=int)
for _ in range(T2):
    cnt[weighted_pick_ares(range(len(W)), W, rng)] += 1
rej, stat, crit = chi2_test(cnt, T2 * p_true)
print(f"A-Res  (k=1): 频率 {cnt / T2}   理论 {p_true}")
print(f"              卡方 {stat:.2f} < {crit} -> 不拒绝 ✅")
assert not rej

rng = np.random.default_rng(SEED + 4)
idx = weighted_sample_replace(W, 200000, rng)
cnt2 = np.bincount(idx, minlength=len(W))
rej2, stat2, crit2 = chi2_test(cnt2, 200000 * p_true)
print(f"前缀和+二分 : 频率 {cnt2 / 200000}   卡方 {stat2:.2f} < {crit2} -> 不拒绝 ✅")
assert not rej2
print("\n✅ 两条路都严格按权重采样。面试里说清「有放回用前缀和、流式无放回用 A-Res」就够了。")

In [ ]:
def weighted_pick_wrong(items, weights, rng):
    """错误版本：按顺序扫描，以 w_i/w_max 的概率"接受"当前元素，取第一个被接受的。
    看起来是"按权重接受"，实际严重偏向靠前的元素。"""
    w = np.asarray(weights, float)
    wmax = w.max()
    while True:                                        # 一轮没接受就再扫一遍
        for i, x in enumerate(items):
            if rng.random() < w[i] / wmax:
                return x

# 手推该版本的真实分布：P(i) ∝ (∏_{j<i}(1 - w_j/wmax)) * (w_i/wmax)
w = np.array(W); surv = np.cumprod(np.r_[1.0, 1 - w[:-1] / w.max()])
p_wrong = surv * (w / w.max()); p_wrong /= p_wrong.sum()

T3 = 50000
rng = np.random.default_rng(SEED + 5)
cnt3 = np.zeros(len(W), dtype=int)
for _ in range(T3):
    cnt3[weighted_pick_wrong(range(len(W)), W, rng)] += 1
print(f"错误版本实测频率 {cnt3 / T3}")
print(f"手推预测       {p_wrong}")
print(f"正确的目标     {p_true}")
rej3, stat3, crit3 = chi2_test(cnt3, T3 * p_true)
print(f"\n对照「正比于权重」做卡方: {stat3:.0f} > {crit3} -> 拒绝（严重有偏）✅")
assert rej3 and stat3 > 500
assert np.abs(cnt3 / T3 - p_wrong).max() < 0.01        # 与手推分布一致，说明我们理解了它错在哪
# 最刺眼的一点：权重最大的元素反而最少被选到
assert (cnt3 / T3)[-1] < (cnt3 / T3)[0]
print(f"权重最大(w=5)的元素只被选中 {cnt3[-1]/T3:.4f}，比权重最小(w=1)的 {cnt3[0]/T3:.4f} 还少 ——")
print("因为它排在最后，前面四个元素几乎总有一个先被接受。这就是「顺序扫描 + 接受概率」的系统性偏差。")

## 6 · Fisher–Yates 与错误洗牌：先证明必然有偏，再用卡方抓出来

- **正确**：第 $i$ 步从 $[i, n)$ 里取 $j$ 交换。
- **错误**：第 $i$ 步从 $[0, n)$ 里取 $j$ 交换（"每个位置都可能换到，看起来更随机"）。

错误版本共有 $n^n$ 条等概率执行路径，排列有 $n!$ 个。若输出均匀，必须 $n!\mid n^n$。
但对任意 $n\ge3$ 都不成立：由 Bertrand 假设存在素数 $p\in(n/2,\,n]$；
$n$ 为合数时 $p\nmid n$（因 $2p>n$），$n$ 为素数时取 $p=2$。
于是 $n!$ 总含一个不整除 $n$ 的素因子 ⟹ **错误洗牌不可能均匀**。

$n=3$ 时 $27/6$ 不整除，实际是 3 个排列各 $4/27\approx0.148$、另 3 个各 $5/27\approx0.185$。

In [ ]:
from itertools import permutations

for n in range(3, 11):                                  # 整除性论证：一行代码就能否定错误洗牌
    assert (n ** n) % math.factorial(n) != 0, n
print("n^n 不被 n! 整除（n=3..10）:",
      [(n, n ** n % math.factorial(n)) for n in range(3, 8)])

def fy_shuffle(a, rng):
    """Fisher-Yates：第 i 步从 [i, n) 里取 j 交换。"""
    a = list(a)
    n = len(a)
    for i in range(n - 1):
        j = int(rng.integers(i, n))                     # 关键：下界是 i，不是 0
        a[i], a[j] = a[j], a[i]
    return a

def naive_shuffle(a, rng):
    """错误版本：每步换到任意位置 [0, n)。"""
    a = list(a)
    n = len(a)
    for i in range(n):
        j = int(rng.integers(0, n))
        a[i], a[j] = a[j], a[i]
    return a

PN, T4 = 3, 60000                                       # PN 而不是 n：后面的 cell 会用到它
perms = list(permutations(range(PN)))                   # 6 个排列 -> 自由度 5
pos = {p: i for i, p in enumerate(perms)}

def perm_counts(fn, seed):
    rng = np.random.default_rng(seed)
    c = np.zeros(len(perms), dtype=int)
    for _ in range(T4):
        c[pos[tuple(fn(range(PN), rng))]] += 1
    return c

c_fy, c_naive = perm_counts(fy_shuffle, SEED + 6), perm_counts(naive_shuffle, SEED + 7)
exp = np.full(len(perms), T4 / len(perms))
r1, s1, cr = chi2_test(c_fy, exp)
r2, s2, _ = chi2_test(c_naive, exp)
print(f"\n排列          {perms}")
print(f"Fisher-Yates  {c_fy}  卡方 {s1:.2f} < {cr}  -> 不拒绝均匀 ✅")
print(f"错误洗牌      {c_naive}  卡方 {s2:.1f} > {cr}  -> 拒绝（有偏）✅")
print(f"错误版本频率  {c_naive / T4}")
print(f"理论 4/27={4/27:.4f} 或 5/27={5/27:.4f}，均匀应为 {1/6:.4f}")
assert not r1 and r2 and s2 > 200
# 每个频率都应贴近 4/27 或 5/27 之一，而不是 1/6
for f in c_naive / T4:
    assert min(abs(f - 4 / 27), abs(f - 5 / 27)) < 0.006, f
print("\n✅ 理论预测的 4/27 与 5/27 被实测精确复现 —— 偏差不是噪声，是结构性的。")

In [ ]:
# n=4 的"位置 x 取值"频率矩阵：偏差集中在哪里？
n2, T5 = 4, 40000

def pos_value_freq(fn, seed):
    rng = np.random.default_rng(seed)
    M = np.zeros((n2, n2))
    for _ in range(T5):
        out = fn(range(n2), rng)
        for p, v in enumerate(out):
            M[p, v] += 1
    return M / T5

F_fy, F_naive = pos_value_freq(fy_shuffle, SEED + 8), pos_value_freq(naive_shuffle, SEED + 9)
print("Fisher-Yates 的 P(位置 p 上是值 v)（理论全为 0.25）:\n", F_fy)
print("\n错误洗牌:\n", F_naive)
d_fy, d_bad = np.abs(F_fy - 0.25).max(), np.abs(F_naive - 0.25).max()
print(f"\n最大偏差: Fisher-Yates {d_fy:.4f}   错误洗牌 {d_bad:.4f}（相差 {d_bad/d_fy:.0f} 倍）")
assert d_fy < 0.012          # 40000 次下 1 sigma≈0.0022，0.012 约 5.5 sigma，固定种子稳定通过
assert d_bad > 0.02
worst = np.unravel_index(np.argmax(np.abs(F_naive - 0.25)), F_naive.shape)
print(f"错误洗牌偏得最狠的格子: 位置 {worst[0]} 上是值 {worst[1]}，"
      f"频率 {F_naive[worst]:.4f} vs 0.25")
print("\n可迁移结论：**「看起来更随机」不是论证**。判断一个随机算法对不对，"
      "\n先数执行路径（整除性），再做卡方——两者都不需要读别人的实现。")

## 7 · 9 道 timed drill：按真实节奏做一遍

**用法**（请真的按计时器做，这一节的价值全在「计时」两个字上）：

1. `print_drill(k)` —— 打印题面、建议用时、频率/档位、**评分要点**（答案不会出现）
2. 开一个新 cell，按建议用时写你的实现；写的时候**出声**（对着手机录音，之后回听）
3. `check(k)` —— 用参考实现跑对拍与边界测试；`reveal(k)` —— 才打印答案源码
4. 用 §8 的 `grade()` 给自己打分，再用 `audit()` 查七个翻车点

九道题覆盖 ML/CV 岗高频清单：矩阵旋转、螺旋遍历、区间合并、Top-K、滑窗最大值、
字符串解析、流式蓄水池、二维矩阵搜索、随机打乱。
**检测专项白板题（IoU / NMS / mAP / 匈牙利 / Focal）见 C61-05，本模块不重复。**

In [ ]:
# ── 对拍基线：每道题的"暴力解"。面试里它是你的正确性基线，这里它是测试的裁判 ──
def spiral_brute(mat):
    # 按方向模拟 + visited：慢但一定对，用来给螺旋遍历当裁判
    if not mat or not mat[0]:
        return []
    R, C = len(mat), len(mat[0])
    seen = [[False] * C for _ in range(R)]
    dr, dc = (0, 1, 0, -1), (1, 0, -1, 0)
    r = c = d = 0
    out = []
    for _ in range(R * C):
        out.append(mat[r][c])
        seen[r][c] = True
        nr, nc = r + dr[d], c + dc[d]
        if not (0 <= nr < R and 0 <= nc < C and not seen[nr][nc]):
            d = (d + 1) % 4                        # 撞墙或撞已访问 -> 右转
            nr, nc = r + dr[d], c + dc[d]
        r, c = nr, nc
    return out

def merge_brute(intervals):
    # O(n^3) 的不动点合并：只要还有两个区间**真重叠**就合并成它们的凸包，直到不再变化。
    # 慢，但它直接照抄"真重叠"的定义（半开区间：接触不算），所以可以当裁判。
    iv = [list(x) for x in intervals]
    changed = True
    while changed:
        changed = False
        for i in range(len(iv)):
            for j in range(i + 1, len(iv)):
                if iv[i][0] < iv[j][1] and iv[j][0] < iv[i][1]:      # 交集长度 > 0
                    iv[i] = [min(iv[i][0], iv[j][0]), max(iv[i][1], iv[j][1])]
                    iv.pop(j)
                    changed = True
                    break
            if changed:
                break
    return sorted(tuple(x) for x in iv)

def window_max_brute(a, k):
    k = min(k, len(a)) if a else k
    return [max(a[i:i + k]) for i in range(len(a) - k + 1)] if a and k > 0 else []

def topk_brute(items, k):
    cnt = Counter(items)
    return [x for x, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:k]]

print("✅ 四个暴力裁判就位（螺旋模拟 / 点集合并 / 逐窗口 max / 全排序 Top-K）。")
assert spiral_brute([[1, 2], [3, 4]]) == [1, 2, 4, 3]
assert merge_brute([(1, 3), (2, 5)]) == [(1, 5)]
assert window_max_brute([1, 3, 2], 2) == [3, 3]
assert topk_brute("aabbbc", 2) == ["b", "a"]

In [ ]:
DRILLS = {}

DRILLS[1] = dict(
    title="原地旋转矩阵 90°（顺时针）", minutes=8, freq="高频", level="必会",
    prompt="给一个 n x n 的二维列表，**原地**顺时针旋转 90 度。不允许新开一个 n x n 数组。",
    points=["先问「一定是方阵吗」「必须原地吗」——非方阵无法原地",
            "说出分解：**先沿主对角线转置，再左右翻转每一行**（或先上下翻转再转置）",
            "转置只走上三角 j > i，否则每对元素被交换两次等于没换",
            "n=0 / n=1 / n 为奇数（中心元素不动）三个边界要提一句",
            "复杂度 O(n^2) 时间、O(1) 额外空间——把「额外」这个词说出来"],
    code=r'''
def rotate90(mat):
    # 原地顺时针旋转：先沿主对角线转置，再左右翻转每一行
    n = len(mat)
    for i in range(n):
        for j in range(i + 1, n):            # 只走上三角，否则交换两次等于没换
            mat[i][j], mat[j][i] = mat[j][i], mat[i][j]
    for row in mat:
        row.reverse()
    return mat
''',
    test=r'''
assert rotate90([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
assert rotate90([]) == [] and rotate90([[1]]) == [[1]]
m = [[1, 2], [3, 4]]
assert rotate90(m) is m and m == [[3, 1], [4, 2]]      # 必须原地：返回的就是同一个对象
for n in range(1, 8):
    a = np.arange(n * n).reshape(n, n)
    assert rotate90(a.tolist()) == np.rot90(a, k=-1).tolist(), n   # k=-1 是顺时针
''')

DRILLS[2] = dict(
    title="螺旋遍历矩阵（m x n）", minutes=10, freq="高频", level="必会",
    prompt="给一个 m x n 的二维列表，按顺时针螺旋顺序返回所有元素组成的一维列表。",
    points=["用**四个边界** top/bottom/left/right 收缩，比方向数组更容易讲清楚",
            "**只剩一行或一列时的退化**是唯一考点：回扫前必须再判一次 top <= bottom / left <= right",
            "空矩阵与 [[]] 都要能过（mat 为空 或 mat[0] 为空）",
            "复杂度 O(mn)，每个元素只被访问一次——说出「每次收缩后边界严格变窄，所以循环一定终止」",
            "如果卡住：先写 1 x n 与 m x 1 两个特例，再合并"],
    code=r'''
def spiral_order(mat):
    # 四个边界向内收缩；每收缩一次就重新判一次是否已经越过
    if not mat or not mat[0]:
        return []
    top, bottom = 0, len(mat) - 1
    left, right = 0, len(mat[0]) - 1
    out = []
    while top <= bottom and left <= right:
        for c in range(left, right + 1):          # 上边：左 -> 右
            out.append(mat[top][c])
        top += 1
        for r in range(top, bottom + 1):          # 右边：上 -> 下
            out.append(mat[r][right])
        right -= 1
        if top <= bottom:                         # 只剩一行时不能再回扫
            for c in range(right, left - 1, -1):
                out.append(mat[bottom][c])
            bottom -= 1
        if left <= right:                         # 只剩一列时不能再上扫
            for r in range(bottom, top - 1, -1):
                out.append(mat[r][left])
            left += 1
    return out
''',
    test=r'''
assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([]) == [] and spiral_order([[]]) == []
assert spiral_order([[1, 2, 3, 4]]) == [1, 2, 3, 4]          # 单行
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]            # 单列
_r = np.random.default_rng(101)
for _ in range(120):
    m, n = int(_r.integers(1, 7)), int(_r.integers(1, 7))
    M = _r.integers(0, 99, (m, n)).tolist()
    got = spiral_order(M)
    assert got == spiral_brute(M), (M, got)
    assert sorted(got) == sorted(v for row in M for v in row)   # 不重不漏
''')

DRILLS[3] = dict(
    title="区间合并", minutes=8, freq="高频", level="必会",
    prompt="给若干半开区间 [s, e)，合并所有**真重叠**的区间并按起点升序返回。"
           "约定：接触（前一个的 e 恰好等于后一个的 s）**不**合并。",
    points=["第一句就问「区间是开还是闭」「接触算不算重叠」——这是本题唯一的歧义点",
            "排序键是**起点**；说出「排序后只需与结果里的最后一个比较」这条不变量",
            "合并时是 max(前一个的 e, 当前 e)，不能直接用当前 e（当前可能被完全包含）",
            "空输入返回 []；单区间原样返回",
            "O(n log n) 由排序主导——不要说成 O(n)"],
    code=r'''
def merge_intervals(intervals):
    # 按起点排序后线性扫描；out[-1] 是"当前正在生长的区间"
    if not intervals:
        return []
    out = []
    for s, e in sorted(intervals):
        if out and s < out[-1][1]:                # 严格小于 = 真重叠（接触不算）
            out[-1][1] = max(out[-1][1], e)       # 当前区间可能被完全包含
        else:
            out.append([s, e])
    return [tuple(x) for x in out]
''',
    test=r'''
assert merge_intervals([]) == []
assert merge_intervals([(1, 4), (2, 5)]) == [(1, 5)]
assert merge_intervals([(1, 3), (3, 5)]) == [(1, 3), (3, 5)]     # 接触不合并
assert merge_intervals([(5, 7), (1, 3), (2, 6)]) == [(1, 7)]
assert merge_intervals([(1, 10), (2, 3)]) == [(1, 10)]           # 完全包含
_r = np.random.default_rng(102)
for _ in range(200):
    iv = [(int(s), int(s) + int(_r.integers(1, 6)))
          for s in _r.integers(0, 20, int(_r.integers(0, 9)))]
    assert merge_intervals(iv) == merge_brute(iv), iv
''')

print(f"已登记 {len(DRILLS)} 道 drill（矩阵旋转 / 螺旋遍历 / 区间合并）")

In [ ]:
DRILLS[4] = dict(
    title="Top-K 高频元素", minutes=10, freq="高频", level="必会",
    prompt="给一个元素序列和整数 k，返回出现次数最多的 k 个元素。"
           "次数相同时按元素本身升序，保证结果**确定**。",
    points=["先问「k 会大于不同元素个数吗」「次数相同怎么排」——tie-break 不定就是不确定的输出",
            "两种解法都要说：全排序 O(n log n) vs 大小为 k 的堆 O(n log k)",
            "n 很大而 k 很小时堆才有意义；k 接近 n 时全排序反而更快（常数小）",
            "确定性 tie-break 的写法：排序键用 (-次数, 元素)",
            "延伸：数据流场景要用 Count-Min Sketch 或 Space-Saving（说出名字即可）"],
    code=r'''
def top_k_frequent(items, k):
    # 全排序版：键 (-次数, 元素) 保证 tie-break 确定
    cnt = Counter(items)
    return [x for x, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:k]]

def top_k_frequent_heap(items, k):
    # 堆版 O(n log k)：nsmallest 对 (-次数, 元素) 取最小，等价于次数最多、元素最小优先
    cnt = Counter(items)
    return [x for _, x in heapq.nsmallest(k, ((-c, x) for x, c in cnt.items()))]
''',
    test=r'''
assert top_k_frequent("aabbbc", 2) == ["b", "a"]
assert top_k_frequent([], 3) == [] and top_k_frequent("abc", 10) == ["a", "b", "c"]
assert top_k_frequent("abab", 1) == ["a"]                # 次数相同 -> 取元素更小的
_r = np.random.default_rng(103)
for _ in range(200):
    xs = _r.integers(0, 8, int(_r.integers(0, 40))).tolist()
    k = int(_r.integers(0, 9))
    ref = topk_brute(xs, k)
    assert top_k_frequent(xs, k) == ref, (xs, k)
    assert top_k_frequent_heap(xs, k) == ref, (xs, k)    # 两种解法必须逐位一致
''')

DRILLS[5] = dict(
    title="滑动窗口最大值", minutes=12, freq="中频", level="必会",
    prompt="给数组 a 与窗口长度 k，返回每个长度为 k 的窗口内的最大值组成的列表。要求 O(n)。",
    points=["**不变量**要说出来：双端队列里存下标，对应的值单调递减，队首恒为当前窗口最大值",
            "为什么均摊 O(n)：每个下标最多入队一次、出队一次",
            "两次弹出的区别：尾部弹「比新元素小的」，头部弹「已滑出窗口的」",
            "k > len(a) 或 k <= 0 的边界要处理（面试官很爱追这个）",
            "延伸：改成滑窗最小值只需反转比较符；同时要最大最小就维护两个队列"],
    code=r'''
def sliding_window_max(a, k):
    # 单调递减双端队列存**下标**。不变量：a[dq[0]] 是当前窗口最大值
    if not a or k <= 0:
        return []
    k = min(k, len(a))
    dq, out = deque(), []
    for i, x in enumerate(a):
        while dq and a[dq[-1]] <= x:      # 比新元素小的永远不可能再当最大值
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                # 队首已滑出窗口
            dq.popleft()
        if i >= k - 1:
            out.append(a[dq[0]])
    return out
''',
    test=r'''
assert sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert sliding_window_max([], 3) == [] and sliding_window_max([1, 2], 0) == []
assert sliding_window_max([2, 1], 5) == [2]                  # k 超长 -> 退化成全局最大
assert sliding_window_max([5, 5, 5], 2) == [5, 5]            # 全相同（<= 保证不死循环）
_r = np.random.default_rng(104)
for _ in range(300):
    a = _r.integers(-9, 9, int(_r.integers(1, 25))).tolist()
    k = int(_r.integers(1, 8))
    assert sliding_window_max(a, k) == window_max_brute(a, k), (a, k)
''')

DRILLS[6] = dict(
    title="解析检测记录（字符串解析）", minutes=10, freq="中频", level="必会",
    prompt="解析形如 `cls=3;score=0.87;box=10,20,30,40` 的记录行，返回 (records, n_bad)。"
           "空行与以 # 开头的注释行跳过且不计入 n_bad；**脏行只计数，不抛异常**。",
    points=["先问清脏数据策略：抛异常 / 跳过 / 用默认值——**真实的标注文件一定有脏行**",
            "字段缺失、分隔符错、box 不是 4 个数，三类都要覆盖",
            "用 split('=', 1) 而不是 split('=')，值里可能含 =",
            "不要用 str += 拼接（O(n^2)）；不要用正则解决一切（可读性差且难解释）",
            "报复杂度时说「O(总字符数)」比「O(n)」准确"],
    code=r'''
def parse_records(lines):
    # 脏行只计数不中断：真实标注文件里脏行是常态，抛异常会让整个数据管线停摆
    records, n_bad = [], 0
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue                                       # 空行/注释不算脏数据
        try:
            fields = dict(part.split("=", 1) for part in line.split(";"))
            box = tuple(float(v) for v in fields["box"].split(","))
            if len(box) != 4:
                raise ValueError("box 必须是 4 个数")
            records.append(dict(cls=int(fields["cls"]),
                                score=float(fields["score"]), box=box))
        except (ValueError, KeyError):
            n_bad += 1
    return records, n_bad
''',
    test=r'''
LINES = ["cls=3;score=0.87;box=10,20,30,40",
         "# 注释", "", "   ",
         "cls=1;score=0.5;box=0,0,5,5",
         "cls=x;score=0.5;box=0,0,5,5",        # cls 不是整数
         "cls=2;score=0.9;box=1,2,3",          # box 只有 3 个数
         "cls=2 score=0.9 box=1,2,3,4",        # 分隔符错 -> 缺 box 字段
         "score=0.9;box=1,2,3,4"]              # 缺 cls
recs, bad = parse_records(LINES)
assert (len(recs), bad) == (2, 4), (len(recs), bad)
assert recs[0] == dict(cls=3, score=0.87, box=(10.0, 20.0, 30.0, 40.0))
assert parse_records([]) == ([], 0)
assert parse_records(["# only comment", ""]) == ([], 0)      # 注释不该被算成脏行
''')

print(f"已登记 {len(DRILLS)} 道 drill（+ Top-K / 滑窗最大值 / 字符串解析）")

In [ ]:
DRILLS[7] = dict(
    title="流式蓄水池采样（不能用 len）", minutes=10, freq="中频", level="加分",
    prompt="从一个**只能遍历一次、长度未知**的可迭代对象里等概率取 k 个元素，"
           "返回 (pool, n_seen)。禁止先转成 list，也禁止调用 len(stream)。",
    points=["先说出目标：每个元素最终留下的概率都是 k/n，且内存只有 O(k)",
            "证明的关键是**裂项**：第 t 个元素踢掉某个特定位置的概率是 1/(t+1)，存活是 t/(t+1)，连乘抵消",
            "n < k 时要返回全部元素（面试官常拿这个当边界追问）",
            "面试官会问「怎么验证你写对了」——答：**做频率检验或卡方检验**，这题的正确性是概率陈述",
            "延伸：Vitter 的 Z 算法用跳跃分布把随机数调用降到 O(k log(n/k))；加权版是 A-Res"],
    code=r'''
def reservoir_k_stream(stream, k, rng):
    # 只过一遍、不知道总长；n 边遍历边自增，所以用 rng.integers(0, n) 等价于 k/(i+1)
    pool, n = [], 0
    for x in stream:
        n += 1
        if len(pool) < k:
            pool.append(x)                       # 前 k 个直接入池
        else:
            j = int(rng.integers(0, n))          # n 已经含当前元素
            if j < k:
                pool[j] = x
    return pool, n
''',
    test=r'''
_r = np.random.default_rng(105)
pool, _n = reservoir_k_stream((x for x in range(7)), 10, _r)  # 生成器 + n < k
assert sorted(pool) == list(range(7)) and _n == 7
pool, _n = reservoir_k_stream(iter([]), 3, _r)
assert pool == [] and _n == 0
N_, K_, T_ = 9, 3, 12000
cnt = np.zeros(N_)
_r = np.random.default_rng(106)
for _ in range(T_):
    p, _n = reservoir_k_stream((x for x in range(N_)), K_, _r)
    assert _n == N_ and len(p) == K_ and len(set(p)) == K_    # 无放回：不能有重复
    for x in p:
        cnt[x] += 1
freq = cnt / T_
assert abs(freq.mean() - K_ / N_) < 1e-12                     # 每轮恰好取 K 个
assert np.abs(freq - K_ / N_).max() < 0.022, freq             # 约 5 sigma
''')

DRILLS[8] = dict(
    title="有序二维矩阵搜索（阶梯法）", minutes=8, freq="高频", level="必会",
    prompt="矩阵每行从左到右递增、每列从上到下递增。查找 target，"
           "返回任意一个 (行, 列)，不存在返回 None。要求 O(m + n)。",
    points=["**从右上角（或左下角）出发**是全部考点：那里是唯一「两个方向单调性相反」的角",
            "说清每一步排除了什么：当前值 > target 就整列排除，< target 就整行排除",
            "为什么不能从左上角开始：两个方向都变大，无法二选一",
            "为什么不是 O(log(mn))：这个矩阵**不是全序**的，二分需要严格的一维有序",
            "边界：空矩阵、[[]]、target 比最小值还小 / 比最大值还大"],
    code=r'''
def search_sorted_matrix(mat, target):
    # 从右上角走阶梯：每一步排除一整行或一整列，所以最多 m + n 步
    if not mat or not mat[0]:
        return None
    r, c = 0, len(mat[0]) - 1
    while r < len(mat) and c >= 0:
        v = mat[r][c]
        if v == target:
            return (r, c)
        if v > target:
            c -= 1                    # 该列下方都更大 -> 整列排除
        else:
            r += 1                    # 该行左侧都更小 -> 整行排除
    return None
''',
    test=r'''
assert search_sorted_matrix([], 5) is None and search_sorted_matrix([[]], 5) is None
assert search_sorted_matrix([[1, 3], [2, 4]], 3) == (0, 1)
assert search_sorted_matrix([[1, 3], [2, 4]], 5) is None
_r = np.random.default_rng(107)
for _ in range(80):
    m, n = int(_r.integers(1, 7)), int(_r.integers(1, 7))
    M = np.cumsum(np.cumsum(_r.integers(1, 4, (m, n)), axis=0), axis=1).tolist()
    vals = {v for row in M for v in row}
    for t in vals:
        rc = search_sorted_matrix(M, t)
        assert rc is not None and M[rc[0]][rc[1]] == t, (M, t)
    for t in (min(vals) - 1, max(vals) + 1):
        assert search_sorted_matrix(M, t) is None, (M, t)
''')

DRILLS[9] = dict(
    title="随机打乱（Fisher–Yates）", minutes=5, freq="中频", level="必会",
    prompt="原地等概率打乱一个列表，返回同一个对象。不允许用 rng.shuffle / random.shuffle。",
    points=["第 i 步必须从 **[i, n)**（或倒着走时的 [0, i]）里取，不是 [0, n)",
            "说出错误版本为什么必然有偏：n^n 条等概率路径，而 n! 不整除 n^n",
            "n=0 / n=1 要能过；返回的必须是同一个对象（原地）",
            "补一句 C/C++ 里的 modulo bias：rand() % n 在 n 不整除 2^32 时有偏，要用拒绝采样",
            "面试官问「怎么验证」——答：对小 n 做全排列的卡方检验"],
    code=r'''
def fisher_yates(a, rng):
    # 从后往前：第 i 步从 [0, i] 里取 j 交换。下界/上界写错就会有偏
    for i in range(len(a) - 1, 0, -1):
        j = int(rng.integers(0, i + 1))
        a[i], a[j] = a[j], a[i]
    return a
''',
    test=r'''
_r = np.random.default_rng(108)
a = list(range(6))
assert fisher_yates(a, _r) is a and sorted(a) == list(range(6))   # 原地 + 是排列
assert fisher_yates([], _r) == [] and fisher_yates([7], _r) == [7]
T_ = 60000
_r = np.random.default_rng(109)
c = np.zeros(6, dtype=int)
for _ in range(T_):
    c[pos[tuple(fisher_yates(list(range(3)), _r))]] += 1
rej, stat, crit = chi2_test(c, np.full(6, T_ / 6))
assert not rej, (c, stat, crit)                                   # 不该被拒绝
''')

assert len(DRILLS) == 9
print(f"已登记 {len(DRILLS)} 道 drill（+ 流式蓄水池 / 二维矩阵搜索 / 随机打乱）")
print(f"建议总用时 {sum(d['minutes'] for d in DRILLS.values())} 分钟 "
      f"= {sum(d['minutes'] for d in DRILLS.values())/45:.1f} 场 45 分钟面试的写码时间")

In [ ]:
def print_drill(k):
    d = DRILLS[k]
    print(f"── Drill {k} · {d['title']}")
    print(f"   {d['freq']} · {d['level']} · 建议用时 {d['minutes']} 分钟")
    print(f"   题面: {d['prompt']}")
    print("   评分要点（写完自己逐条打勾）:")
    for i, p in enumerate(d["points"], 1):
        print(f"     {i}. {p}")

def reveal(k):
    print(DRILLS[k]["code"].strip())              # 只有主动调用才会看到答案

def check(k, quiet=False):
    exec(DRILLS[k]["code"], globals())            # 让参考实现生效
    t0 = time.perf_counter()
    exec(DRILLS[k]["test"], globals())            # 对拍 + 边界 + （采样题）统计检验
    if not quiet:
        print(f"✅ Drill {k} 《{DRILLS[k]['title']}》测试通过  "
              f"({time.perf_counter()-t0:.2f}s)")

print_drill(5)                                    # 示范：看题不看答案
print()
for k in sorted(DRILLS):
    assert len(DRILLS[k]["points"]) >= 4, k       # 每题至少 4 条评分要点
    assert DRILLS[k]["freq"] in ("高频", "中频", "低频")
    assert DRILLS[k]["level"] in ("必会", "加分")
    check(k)
must = [k for k in DRILLS if DRILLS[k]["level"] == "必会"]
print(f"\n九道全部通过。其中「必会」{len(must)} 道（{sorted(must)}）——先把这些练到"
      f"能带解说在建议用时内写完，再碰「加分」项。")
assert len(must) >= 6

## 8 · 五维评分表代码化：为什么「全对但不说不测」会挂

权重：正确性 30% · 复杂度 20% · 沟通 20% · 代码质量 15% · 测试意识 15%。
**正确性只占 30%，沟通 + 测试合起来 35%。**

下面把评分表写成函数，然后把 §3 的审计器接进来——**翻车点会给对应维度设上限**，
这就解释了为什么「代码全对但全程沉默、写完不测」的总分会低于「代码 75% 但全程解说并自测」。

In [ ]:
RUBRIC = [("正确性", 0.30), ("复杂度分析", 0.20), ("沟通", 0.20),
          ("代码质量", 0.15), ("测试意识", 0.15)]
TIERS = [(80, "strong hire"), (65, "hire（及格线）"), (50, "lean no"), (0, "no hire")]
assert abs(sum(w for _, w in RUBRIC) - 1.0) < 1e-12

def grade(scores):
    """scores: {维度: 0..4} -> (总分 0-100, 档位)。"""
    assert set(scores) == {d for d, _ in RUBRIC}, "五个维度都要打分，缺一项就是缺证据"
    assert all(0 <= v <= 4 for v in scores.values())
    total = sum(w * scores[d] for d, w in RUBRIC) / 4 * 100
    tier = next(t for cut, t in TIERS if total >= cut)
    return round(total, 1), tier

CASES = {
    "A 全都不错":          dict(正确性=4, 复杂度分析=3, 沟通=4, 代码质量=3, 测试意识=3),
    "B 代码75%+全程解说":  dict(正确性=3, 复杂度分析=3, 沟通=3, 代码质量=2, 测试意识=2),
    "C 代码全对但不说不测": dict(正确性=4, 复杂度分析=2, 沟通=1, 代码质量=2, 测试意识=0),
    "D 沟通好但思路错":    dict(正确性=1, 复杂度分析=1, 沟通=4, 代码质量=2, 测试意识=1),
}
for name, sc in CASES.items():
    print(f"{name:<22} {grade(sc)}")

assert grade(CASES["B 代码75%+全程解说"]) == (67.5, "hire（及格线）")
assert grade(CASES["C 代码全对但不说不测"]) == (52.5, "lean no")
assert grade(CASES["B 代码75%+全程解说"])[0] > grade(CASES["C 代码全对但不说不测"])[0]
gap = grade(CASES["B 代码75%+全程解说"])[0] - grade(CASES["C 代码全对但不说不测"])[0]
print(f"\n**代码写完 75% 但全程解说 + 自测（67.5）比代码全对却不说不测（52.5）高 {gap} 分。**")
print("这不是评分表设计得奇怪：后者提供的可迁移信息太少，面试官无法预测你面对新题会怎么做。")

weakest = min(CASES["C 代码全对但不说不测"], key=lambda d: CASES["C 代码全对但不说不测"][d])
print(f"C 最该补的一项: {weakest}（补到 3 分 -> "
      f"{grade({**CASES['C 代码全对但不说不测'], weakest: 3})}）")
assert weakest == "测试意识"
assert grade({**CASES["C 代码全对但不说不测"], "测试意识": 3})[0] >= 63.5

In [ ]:
# 把 §3 的审计器接进评分表：每个翻车点给一个维度**设上限**（不是扣分，是封顶）
CAPS = {"①一上来就写": ("沟通", 1), "②不问边界": ("正确性", 3), "③写完不测": ("测试意识", 0),
        "④复杂度说错": ("复杂度分析", 1), "⑤变量名混乱": ("代码质量", 1),
        "⑥默默改需求": ("正确性", 1), "⑦沉默超30秒": ("沟通", 1)}
assert set(CAPS) == set(PITFALLS)

def grade_with_audit(self_scores, session):
    """自评分 + 录屏审计 -> (总分, 档位, 被封顶后的分数, 命中的翻车点)。"""
    s = dict(self_scores)
    hits = audit(session)
    for p in hits:
        dim, cap = CAPS[p]
        s[dim] = min(s[dim], cap)                  # 取 min：多个翻车点作用于同一维度时取最严
    return (*grade(s), s, hits)

optimistic = dict(正确性=4, 复杂度分析=4, 沟通=4, 代码质量=4, 测试意识=4)
tot_b, tier_b, capped_b, hits_b = grade_with_audit(optimistic, session_bad)
tot_g, tier_g, capped_g, hits_g = grade_with_audit(optimistic, session_good)
print(f"自评全 4 分（= 100）：")
print(f"  翻车场次: 命中 {len(hits_b)}/7 -> 封顶后 {capped_b} -> {tot_b} / {tier_b}")
print(f"  达标场次: 命中 {len(hits_g)}/7 -> {tot_g} / {tier_g}")
assert grade(optimistic) == (100.0, "strong hire")
assert tot_b == 21.2 and tier_b == "no hire", (tot_b, tier_b)
assert tot_g == 100.0

print("\n单点修复的收益（在翻车场次的基础上，只改掉一项）:")
for p in sorted(hits_b):
    fake = {k: v for k, v in session_bad.items()}
    if p == "③写完不测":
        fake["tests_run"] = 4
    elif p == "②不问边界":
        fake["clarifications"] = ["规模", "边界", "空输入"]
    elif p == "①一上来就写":
        fake["first_keystroke_sec"] = 60 + 13 * 60
    elif p == "⑦沉默超30秒":
        fake["speech_times"] = good_run
    elif p == "④复杂度说错":
        fake["complexity_claims"] = {"sorted(a)": "O(n log n)"}
    elif p == "⑤变量名混乱":
        fake["code"] = "left = 0\nright = 0\ncount_by_class = {}\n"
    else:
        fake["impl"] = dict(demo_spec)
    print(f"  修掉 {p}: {tot_b} -> {grade_with_audit(optimistic, fake)[0]}")
assert grade_with_audit(optimistic, {**session_bad, "tests_run": 4})[0] > tot_b
# 注意 ① 和 ⑦ 都封顶「沟通」这一维 —— 只修一个总分不动，必须两个一起修
assert grade_with_audit(optimistic, {**session_bad, "first_keystroke_sec": 60 + 13 * 60})[0] == tot_b
assert grade_with_audit(optimistic, {**session_bad, "first_keystroke_sec": 60 + 13 * 60,
                                     "speech_times": good_run})[0] > tot_b
print("\n注意 ① 与 ⑦ 都封顶「沟通」：只修一个总分不动（21.2 -> 21.2），两个一起修才有效 ——")
print("这正是现实里的样子：**沟通分不是靠单个动作拿的，是靠整场的一致性拿的**。")
print("\n可迁移结论：**这七项里没有一项需要提升算法能力。**"
      "\n它们的修复成本是分钟级的，而收益是整档的。")

---
## ✏️ 练习 1：分层蓄水池采样（中频 · 加分 · 直接对应长尾数据抽样）

实现 `stratified_reservoir(stream, k_per_class, rng)`：

- `stream` 产出 `(cls, item)` 二元组，**长度未知、只能过一遍**
- 每个类别独立维护一个大小 `k_per_class` 的蓄水池
- 返回 `{cls: [items...]}`；某类样本数不足 `k_per_class` 时**全部保留**

这就是长尾数据集抽可视化子集 / 抽评测子集的标准做法（见 C58-02）：
**按类分层后再采样，尾部类才不会被整体均匀采样淹没**。

In [ ]:
def stratified_reservoir(stream, k_per_class, rng):
    # TODO: 每个类别一个蓄水池 + 一个"该类已见过多少个"的计数器
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ITEMS = ([(0, i) for i in range(100)]           # 头部类 100 个
         + [(1, 1000 + i) for i in range(30)]   # 中部类 30 个
         + [(2, 2000 + i) for i in range(4)])   # 尾部类只有 4 个
KPC = 5
pools = stratified_reservoir(iter(ITEMS), KPC, np.random.default_rng(41))
assert set(pools) == {0, 1, 2}
assert len(pools[0]) == KPC and len(pools[1]) == KPC
assert sorted(pools[2]) == [2000, 2001, 2002, 2003]     # 尾部类不足 k -> 全部保留
assert all(x // 1000 == c for c in pools for x in pools[c])   # 不能串类

# 类内均匀性：头部类 100 个元素的入选频率都应 ≈ k/n = 0.05
T_ = 4000
cnt = np.zeros(100)
rr = np.random.default_rng(43)
for _ in range(T_):
    for x in stratified_reservoir(iter(ITEMS), KPC, rr)[0]:
        cnt[x] += 1
freq = cnt / T_
print(f"头部类入选频率: 均值 {freq.mean():.4f}（理论 {KPC/100:.4f}）"
      f" 最小 {freq.min():.4f} 最大 {freq.max():.4f}")
assert abs(freq.mean() - KPC / 100) < 1e-12      # 每轮恰好取满 k 个 -> 均值是恒等式
assert np.abs(freq - KPC / 100).max() < 0.018    # 约 5 sigma，固定种子下稳定
print("✅ 练习 1 通过：类内均匀 + 尾部类完整保留 + 单次遍历。")
print("   面试加分句：『分层是为了让尾部类的采样概率与它的稀有度解耦——"
      "全局均匀采样时，占 1% 的类在 100 个样本里期望只出现 1 个。』")

## ✏️ 练习 2：加权蓄水池 A-Res，k>1（低频 · 加分 · 一个真实的报告错误）

实现 `weighted_reservoir_k(items, weights, k, rng)`：用 A-Res 做**无放回**加权采样，
给每个元素配 key $u_i^{1/w_i}$，取 key 最大的 $k$ 个。权重 $\le 0$ 的元素永不入选。

自测会验证三件事，第三件是本练习的重点：

1. $k=1$ 时入选概率**严格**正比于权重（卡方不拒绝）
2. **恒等式**：$\sum_i \Pr[i \in S] = k$（因为每次都恰好取 $k$ 个）
3. $k>1$ 时入选概率**不**正比于权重：朴素公式 $k w_i/\sum w_j$ 会给出 **1.6** 这种不可能的数

第 3 条就是长尾重采样里「过采样倍数」被算错的根源（见 C58-03）。

In [ ]:
def weighted_reservoir_k(items, weights, k, rng):
    # TODO: key = u ** (1 / w)，取 key 最大的 k 个；w <= 0 直接跳过
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
IT, WW, KK = list(range(5)), [1.0, 1.0, 1.0, 1.0, 16.0], 2
assert len(weighted_reservoir_k(IT, WW, KK, np.random.default_rng(1))) == KK
assert weighted_reservoir_k(IT, [1.0, 0.0, 0.0, 0.0, 0.0], 1,
                            np.random.default_rng(1)) == [0]      # 权重 0 永不入选

# ① k=1 严格正比于权重
T_ = 30000
p_ref = np.array(WW) / sum(WW)
cnt = np.zeros(5)
rr = np.random.default_rng(45)
for _ in range(T_):
    cnt[weighted_reservoir_k(IT, WW, 1, rr)[0]] += 1
rej, stat, crit = chi2_test(cnt, T_ * p_ref)
print(f"k=1: 频率 {cnt/T_}  理论 {p_ref}  卡方 {stat:.2f} < {crit} ✅")
assert not rej, (stat, crit)

# ② + ③ k=2 的实测入选概率
T2_ = 20000
inc = np.zeros(5)
rr = np.random.default_rng(47)
for _ in range(T2_):
    s = weighted_reservoir_k(IT, WW, KK, rr)
    assert len(set(s)) == KK                      # 无放回
    for x in s:
        inc[x] += 1
p_hat = inc / T2_
p_naive = KK * np.array(WW) / sum(WW)
print(f"k=2 实测入选概率 {p_hat}   和 = {p_hat.sum():.6f}")
print(f"k=2 朴素公式    {p_naive}   <- 最后一项 {p_naive[-1]:.2f} > 1，根本不可能")
assert abs(p_hat.sum() - KK) < 1e-12              # 恒等式：入选概率之和恒等于 k
assert 0.90 < p_hat[-1] < 1.0                     # 最重的那个几乎总入选，但不是必然
assert p_hat[0] > 0.20                            # 朴素公式预测 0.1，实测约 0.26
assert np.abs(p_hat - p_naive).max() > 0.5        # 与朴素公式差得离谱
print("✅ 练习 2 通过。")
print("   面试加分句：『A-Res 在 k=1 时严格按权重，k>1 时只保证入选概率之和为 k；"
      "要精确控制入选概率得用 πps 抽样（Sampford / Tillé）。』")
print("   工程含义：重采样倍数**必须实测**一个 epoch 的类别计数，不能用公式反推。")

## ✏️ 练习 3：Fisher–Yates 洗牌与偏差检验（中频 · 必会）

实现 `shuffle_unbiased(a, rng)`：**原地**等概率打乱并返回同一个对象，
不允许调用 `rng.shuffle` / `random.shuffle`。

自测做三件事：原地性与排列性、$n=0/1$ 边界、以及在 $n=3$ 的全部 6 个排列上做卡方检验
（**你的版本不该被拒绝，而错误洗牌必须被拒绝**）。

In [ ]:
def shuffle_unbiased(a, rng):
    # TODO: 第 i 步只能从 [i, n) 或 [0, i] 里取交换对象；返回 a 本身
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
aa = list(range(6))
out = shuffle_unbiased(aa, np.random.default_rng(51))
assert out is aa and sorted(aa) == list(range(6))         # 原地 + 是一个排列
assert shuffle_unbiased([], np.random.default_rng(51)) == []
assert shuffle_unbiased([7], np.random.default_rng(51)) == [7]

T_ = 60000
rr = np.random.default_rng(53)
c_mine = np.zeros(6, dtype=int)
for _ in range(T_):
    c_mine[pos[tuple(shuffle_unbiased(list(range(3)), rr))]] += 1
rej_mine, s_mine, crit = chi2_test(c_mine, np.full(6, T_ / 6))
c_bad = perm_counts(naive_shuffle, 59)                     # 同样的检验作用在错误版本上
rej_bad, s_bad, _ = chi2_test(c_bad, np.full(6, T4 / 6))
print(f"你的实现  {c_mine}  卡方 {s_mine:.2f} < {crit} -> 不拒绝 ✅")
print(f"错误洗牌  {c_bad}  卡方 {s_bad:.1f} > {crit} -> 拒绝 ✅")
assert not rej_mine, (c_mine, s_mine)
assert rej_bad
# 顺手确认长度 4 时也没有明显偏差
F = pos_value_freq(lambda x, r: shuffle_unbiased(list(x), r), 61)
print(f"n=4 的位置x取值频率最大偏差 {np.abs(F - 0.25).max():.4f}（理论 0）")
assert np.abs(F - 0.25).max() < 0.012
print("✅ 练习 3 通过。")
print("   面试加分句：『错误版本必然有偏，不用做实验：n^n 条等概率路径，而 n! 不整除 n^n。』")

## ✏️ 练习 4：有序矩阵里数出小于 target 的元素个数（高频 · 加分）

实现 `count_less_than(mat, target)`：矩阵每行从左到右递增、每列从上到下递增，
返回**严格小于** target 的元素个数，要求 O(m + n)。

这是 Drill 8 阶梯法的真实变体（也是「第 k 小元素」二分解法的内层）。
提示：从**左下角**出发——那里的两个方向单调性相反，所以每一步都能整行或整列地排除。

In [ ]:
def count_less_than(mat, target):
    # TODO: 从左下角 (m-1, 0) 出发走阶梯，每步累加一整列的一段
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert count_less_than([], 5) == 0 and count_less_than([[]], 5) == 0
assert count_less_than([[1, 3], [2, 4]], 3) == 2          # 1, 2
assert count_less_than([[1, 3], [2, 4]], 1) == 0
assert count_less_than([[1, 3], [2, 4]], 100) == 4
rr = np.random.default_rng(63)
for _ in range(80):
    m, n = int(rr.integers(1, 7)), int(rr.integers(1, 7))
    M = np.cumsum(np.cumsum(rr.integers(1, 4, (m, n)), axis=0), axis=1).tolist()
    flat = [v for row in M for v in row]
    for t in set(flat) | {min(flat) - 1, max(flat) + 1, 0}:
        assert count_less_than(M, t) == sum(v < t for v in flat), (M, t)
print("✅ 练习 4 通过：与逐元素暴力计数在 80 个随机矩阵的所有阈值上一致。")
print("   面试加分句：『这就是「有序矩阵第 k 小」的内层——外层对取值二分，"
      "内层用这个 O(m+n) 的计数，总复杂度 O((m+n) log(值域))。』")

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def stratified_reservoir(stream, k_per_class, rng):
    pools, seen = {}, {}
    for cls, item in stream:
        seen[cls] = seen.get(cls, 0) + 1          # 该类已见过的个数（含当前）
        pool = pools.setdefault(cls, [])
        if len(pool) < k_per_class:
            pool.append(item)                     # 该类还没满 -> 直接放
        else:
            j = int(rng.integers(0, seen[cls]))   # 等价于以 k/seen 的概率替换
            if j < k_per_class:
                pool[j] = item
    return pools

In [ ]:
# 练习 2 参考答案
def weighted_reservoir_k(items, weights, k, rng):
    keyed = []
    for x, w in zip(items, weights):
        if w <= 0:
            continue                              # 权重非正的元素永不入选
        keyed.append((rng.random() ** (1.0 / w), x))   # w 越大，key 越贴近 1
    keyed.sort(reverse=True)                      # 面试里补一句：用大小 k 的最小堆可做到 O(n log k)
    return [x for _, x in keyed[:k]]

In [ ]:
# 练习 3 参考答案
def shuffle_unbiased(a, rng):
    for i in range(len(a) - 1, 0, -1):            # 倒着走：第 i 步从 [0, i] 里取
        j = int(rng.integers(0, i + 1))           # 上界含 i；写成 (0, len(a)) 就是错误洗牌
        a[i], a[j] = a[j], a[i]
    return a                                      # 返回同一个对象 = 原地

In [ ]:
# 练习 4 参考答案
def count_less_than(mat, target):
    if not mat or not mat[0]:
        return 0
    m, n = len(mat), len(mat[0])
    r, c, cnt = m - 1, 0, 0                       # 从左下角出发
    while r >= 0 and c < n:
        if mat[r][c] < target:
            cnt += r + 1                          # 第 c 列的 0..r 行都更小（列自上而下递增）
            c += 1
        else:
            r -= 1                                # 第 c 列的 r..m-1 行都 >= target
    return cnt

---
## 🧪 真实工程胶囊：面试当天可以照抄的一页

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 45 分钟时间预算（打印出来放在手边）
# ══════════════════════════════════════════════════════════════════════
#  0- 3  复述 + 澄清   问三件事：输入规模 / 能否改输入 / 异常输入怎么办
#  3- 6  举例走一遍   5-8 个元素，手算期望输出；同时列边界清单
#  6- 9  暴力解       **只口述不写**，给出复杂度，说"这是我的正确性基线"
#  9-13  最优思路     说清优化依据 + 新复杂度，然后问"我按这个写？"
# 13-30  写代码       签名与返回 -> 主循环 -> 边界。写码窗口只做**翻译**
# 30-38  自测         手跑正例并报中间状态；再跑空/单元素
# 38-45  收尾         时间与空间复杂度 + 一个优化方向 + **反问一个问题**
#
# 超时补救（换部分分，不赌全分）：
#   剩 10 分钟 -> 砍优化，退回一定写得完的版本
#   剩  5 分钟 -> 停手，开始测
#   剩  2 分钟 -> 把已知 bug 说清楚（在哪/为什么/怎么改）+ 报复杂度
#
# ══════════════════════════════════════════════════════════════════════
# B. 句式库（不要直播思维，要播报状态）
# ══════════════════════════════════════════════════════════════════════
# 复述     "我复述一下：给我 __，要我返回 __。对吗？"
# 规模     "输入是 10^3 还是 10^7？这决定我用 O(n^2) 还是必须 O(n log n)。"
# 不变量   "这个 left 指针的不变量是：[left, right) 内没有重复字符。"
# 报缺口   "这一段处理主逻辑，空输入还没处理，我下面补。"
# 卡住-降维 "我先解 k=1 的版本，放开约束后多出来的困难是 __。"
# 卡住-退回 "这条路要求 __ 成立，但反例 __ 说明不成立，我退回 O(n^2)。"
# 接提示   "哦，你是说 __。对，我漏了 __ 这个性质，那么 __ 就可以用 __ 处理。"
# 申请安静 "我需要 30 秒安静想一下这个边界，可以吗？"   <- 沉默必须显式申请
#
# ══════════════════════════════════════════════════════════════════════
# C. 七个翻车点（考前自查，五项与算法能力无关）
# ══════════════════════════════════════════════════════════════════════
# ①一上来就写 ②不问边界 ③写完不测 ④复杂度说错 ⑤变量名混乱 ⑥默默改需求 ⑦沉默>30秒
# 评分权重：正确性 30% / 复杂度 20% / 沟通 20% / 代码质量 15% / 测试意识 15%
# 及格线 65；"代码全对但不说不测" = 52.5（挂），"代码 75% 但全程解说自测" = 67.5（过）
#
# ══════════════════════════════════════════════════════════════════════
# D. debug：先说假设，再验证（禁止 Ctrl+A 重写）
# ══════════════════════════════════════════════════════════════════════
# ① 缩到最小失败用例 -> ② 说出一个**可证伪**的假设 -> ③ 加一行 print 验证
# -> ④ 成立就只改这一处 / 不成立就记下"已排除" -> ⑤ 改完复述根因
# 症状->根因速查：差 1 看循环边界；空输入报错看先访问后判空；
#   小对大错看状态定义；偶发错看重复元素与 tie-break；浮点用 abs(x-y)<1e-9
#
# ══════════════════════════════════════════════════════════════════════
# E. 三种形式的准备动作
# ══════════════════════════════════════════════════════════════════════
# 线上编辑器：提前在同款平台裸写 3 道题（关掉本地 IDE）；背熟常用 API 拼写；写完先按 run
# 白板/纯文本：先规划版面（题面/代码/测试三块），代码区**每行留一行空**；
#              用"手跑状态表"代替运行；允许口头省略 import，但不许省签名与 return
# 共享屏幕：勿扰模式 -> 字号 16-18pt -> 关掉全部无关窗口 -> 桌面与书签栏 -> 麦克风试音
#          专属加分：现场跑你预先准备好的**随机对拍脚本**
#
# ══════════════════════════════════════════════════════════════════════
# F. 采样三题的口述要点（ML/CV 岗的签名题）
# ══════════════════════════════════════════════════════════════════════
# 蓄水池：第 i 个以 k/(i+1) 替换池中随机一个；证明用裂项 prod t/(t+1) = k/n
#         off-by-one（写成 1/i）会让**第一个元素永不入选**，且不报错
# 加权：  有放回 -> 前缀和 + 二分 O(log n)；流式无放回 -> A-Res，key = u^(1/w)
#         k=1 严格正比于权重；**k>1 时入选概率不正比**，只保证 sum = k
#         => 重采样倍数必须**实测**一个 epoch 的类别计数，不能用公式反推
# 洗牌：  第 i 步只能从 [i, n) 取；错误版本 n! 不整除 n^n => 必然有偏
#         C/C++ 里 rand()%n 有 modulo bias，要用拒绝采样
# 验证：  这三题的正确性是概率陈述 -> 用卡方检验或频率收敛验证，期望频数要 > 5
#
# 检测专项白板题（IoU / NMS / mAP / 匈牙利 / Focal）见 C61-05，不在本模块范围
'''
print(RECIPE)
for token in ["13-30", "剩  5 分钟", "申请安静", "⑦沉默>30秒", "52.5", "67.5",
              "可证伪", "modulo bias", "A-Res", "裂项", "C61-05"]:
    assert token in RECIPE, token
print("✅ 一页纸覆盖：时间预算 / 句式库 / 七个翻车点 / debug 流程 / 三种形式 / 采样三题口述要点")

### 小结

- **45 分钟里前 13 分钟不写代码**（澄清 3 + 举例 3 + 暴力解 3 + 最优思路 4），
  写码窗口 17 分钟只做「翻译」，自测 8 分钟。30 分钟是硬检查点：还没跑通就启动三档补救
  （剩 10 分钟降级、剩 5 分钟停手去测、剩 2 分钟把 bug 说清楚）。**补救动作本身不值分，说出来才值分。**
- **评分表是分项的，正确性只占 30%。**本 notebook 用 `grade()` 算出来：
  「代码全对但不说不测」= **52.5（lean no）**，而「代码只写完 75% 但全程解说并自测」= **67.5（hire）**。
  把七个翻车点接进评分表后，自评 100 分的那场实际只有 **21.2**——
  而且 ① 与 ⑦ 同时封顶「沟通」，**只修一个总分不动**：沟通分靠整场一致性拿，不靠单个动作。
- **think aloud 的正确形态是「播报状态」而不是「直播思维」。**16 条句式覆盖澄清/写码/卡住/接提示；
  沉默超过 30 秒必须出声，需要安静就**显式申请**。接提示的四步是：接住 → 归因 → 推进 → 继续写（不要道歉）。
- **debug 要先说一个可证伪的假设，再用一行 print 验证。**跳过假设直接改代码叫「试」，
  它不缩小怀疑区间，还可能引入第二个 bug。永远不要 Ctrl+A 重写——那抹掉了面试官已经给过分的部分。
- **采样三题是少数「正确性可以当场被检验」的题**，也正因此它们是 ML/CV 岗的签名题：
  蓄水池采样写成 `1/i` 而不是 `1/(i+1)` 会让**第一个元素的入选概率变成严格的 0**（卡方统计量 3312，而临界值 29.6），
  但它不报错、不崩、小样本上看不出来；
  「顺序扫描 + 以 w/wmax 接受」这个看起来合理的加权采样，实测让**权重最大的元素反而最少被选到**（0.039 vs 0.200）。
- **Fisher–Yates 的错误版本不需要做实验就能否定**：$n^n$ 条等概率路径不被 $n!$ 整除。
  $n=3$ 时实测精确复现了理论的 $4/27$ 与 $5/27$（而不是 $1/6$），卡方 673 ≫ 20.5；
  $n=4$ 时最大偏差 0.046，是正确版本的 17 倍。**「看起来更随机」永远不是论证。**
- **A-Res 的 $k>1$ 陷阱有真实代价**：入选概率只保证 $\sum_i p_i=k$，并不正比于权重
  （`w=[1,1,1,1,16], k=2` 时朴素公式给出 1.6 这种不可能的数，实测最重元素 0.97、最轻的 0.26）。
  所以**长尾重采样的「过采样倍数」必须实测一个 epoch 的类别计数**，不能用公式反推（见 C58-03）。

至此 C62 五个模块结束：模块 01–04 给你解题的**工具**，模块 05 给你在 45 分钟压力下
**把工具交付出去**的方法。下一步建议按 C63（ML 系统设计）→ C64（ML/DL 知识问答）→
C65（结构化问题求解与沟通）继续，把一面三个板块补齐；检测专项面试题见 C61。